# Day 32 – Decision Trees & Random Forest (Part A + Part B)

Synthetic loan approval dataset with Decision Tree, Random Forest, and Extra Trees.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.inspection import permutation_importance


## Part A – Decision Tree and Random Forest

In [ ]:
rng = np.random.default_rng(32)

n = 2000
annual_income = rng.normal(800000, 250000, n).clip(200000, 2000000)
credit_score = rng.integers(300, 851, n)
loan_amount = rng.normal(500000, 200000, n).clip(100000, 2000000)
employment_years = rng.integers(0, 31, n)
debt_to_income = rng.uniform(0.1, 0.6, n)
num_credit_cards = rng.integers(1, 11, n)

score_raw = 0.000002 * (annual_income - 600000) + (credit_score - 650) / 80 - (debt_to_income - 0.3) * 4 + 0.08 * (employment_years - 3) - 0.000001 * (loan_amount - 400000)
prob = 1 / (1 + np.exp(-score_raw))
noise = rng.normal(0, 0.1, n)
prob_noisy = 1 / (1 + np.exp(-(score_raw + noise)))
approved = (prob_noisy > 0.5).astype(int)

df = pd.DataFrame({
    "annual_income": annual_income.round(0),
    "credit_score": credit_score,
    "loan_amount": loan_amount.round(0),
    "employment_years": employment_years,
    "debt_to_income": debt_to_income.round(3),
    "num_credit_cards": num_credit_cards,
    "approved": approved
})

print(df.head())
print(df["approved"].value_counts(normalize=True))


In [ ]:
X = df[["annual_income", "credit_score", "loan_amount", "employment_years", "debt_to_income", "num_credit_cards"]]
y = df["approved"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=32, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train.shape, X_test.shape)


In [ ]:
dt = DecisionTreeClassifier(max_depth=4, random_state=32)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)
proba_dt = dt.predict_proba(X_test)[:, 1]

acc_dt = accuracy_score(y_test, y_pred_dt)
f1_dt = f1_score(y_test, y_pred_dt)
roc_dt = roc_auc_score(y_test, proba_dt)

print("Decision Tree metrics")
print("Accuracy:", round(acc_dt, 4))
print("F1:", round(f1_dt, 4))
print("ROC-AUC:", round(roc_dt, 4))


In [ ]:
from sklearn.tree import _tree

feature_names = list(X.columns)

leaf_indices = np.where(dt.tree_.children_left == -1)[0]

rules = []

for leaf in leaf_indices:
    path = []
    node = leaf
    while node != 0:
        parent_candidates = np.where((dt.tree_.children_left == node) | (dt.tree_.children_right == node))[0]
        parent = int(parent_candidates[0])
        is_left = dt.tree_.children_left[parent] == node
        feature = feature_names[dt.tree_.feature[parent]]
        threshold = dt.tree_.threshold[parent]
        if is_left:
            cond = f"{feature} <= {threshold:.3f}"
        else:
            cond = f"{feature} > {threshold:.3f}"
        path.append(cond)
        node = parent
    path = path[::-1]
    mask = np.ones(len(X), dtype=bool)
    for cond in path:
        parts = cond.split()
        col = parts[0]
        op = parts[1]
        thr = float(parts[2])
        if op == "<=":
            mask = mask & (X[col].values <= thr)
        else:
            mask = mask & (X[col].values > thr)
    subset_y = y[mask]
    if subset_y.shape[0] == 0:
        continue
    majority = int(subset_y.value_counts().idxmax())
    acc_rule = (subset_y == majority).mean()
    rules.append({"path": path, "majority": majority, "accuracy": acc_rule, "count": subset_y.shape[0]})

rules_sorted = sorted(rules, key=lambda r: r["count"], reverse=True)[:3]

for i, r in enumerate(rules_sorted, 1):
    text = " AND ".join(r["path"])
    label = "APPROVE" if r["majority"] == 1 else "REJECT"
    print(f"Rule {i}: IF {text} -> {label} ({r['accuracy']:.2f} accuracy on {r['count']} samples)")


In [ ]:
param_dist = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 5, 8, 12],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", None]
}

rf_base = RandomForestClassifier(random_state=32, n_jobs=-1)

search = RandomizedSearchCV(rf_base, param_distributions=param_dist, n_iter=15, cv=5, scoring="roc_auc", n_jobs=-1, random_state=32)
search.fit(X_train_scaled, y_train)

rf_best = search.best_estimator_

proba_rf = rf_best.predict_proba(X_test_scaled)[:, 1]
y_pred_rf = rf_best.predict(X_test_scaled)

acc_rf = accuracy_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)
roc_rf = roc_auc_score(y_test, proba_rf)

print("Random Forest best params:")
print(search.best_params_)
print("Random Forest metrics")
print("Accuracy:", round(acc_rf, 4))
print("F1:", round(f1_rf, 4))
print("ROC-AUC:", round(roc_rf, 4))


## Part B – Extra Trees Comparison

In [ ]:
imp_default = rf_best.feature_importances_

perm = permutation_importance(rf_best, X_test_scaled, y_test, scoring="roc_auc", n_repeats=10, random_state=32, n_jobs=-1)
imp_perm = perm.importances_mean

print("Default feature_importances_:")
for name, val in zip(X.columns, imp_default):
    print(name, round(val, 4))

print("
Permutation importance (ROC-AUC change):")
for name, val in zip(X.columns, imp_perm):
    print(name, round(val, 4))


In [ ]:
et = ExtraTreesClassifier(n_estimators=300, max_depth=None, random_state=32, n_jobs=-1)

start_rf = time.time()
rf_best.fit(X_train_scaled, y_train)
rf_time = time.time() - start_rf

start_et = time.time()
et.fit(X_train_scaled, y_train)
et_time = time.time() - start_et

proba_et = et.predict_proba(X_test_scaled)[:, 1]
y_pred_et = et.predict(X_test_scaled)

acc_et = accuracy_score(y_test, y_pred_et)
f1_et = f1_score(y_test, y_pred_et)
roc_et = roc_auc_score(y_test, proba_et)

print("Random Forest time:", round(rf_time, 3), "seconds")
print("Extra Trees time:", round(et_time, 3), "seconds")

print("Extra Trees metrics")
print("Accuracy:", round(acc_et, 4))
print("F1:", round(f1_et, 4))
print("ROC-AUC:", round(roc_et, 4))
